# Molecular filtering: unwanted structures

## Aim
There are some substructures we prefer not to include into our library. We will learn about different types of such unwanted substructures and how to find, highlight and remove them with RDKit.

- Unwanted substructures
- Pan Assay Interference Compounds (PAINS)

## For more details
https://github.com/volkamerlab/teachopencadd/tree/master/teachopencadd/talktorials/T003_compound_unwanted_substructures

## Instructions
Replace XXX with the appropriate code

## Configuration

In [ ]:
# Imports
# 1. Standard library imports
from pathlib import Path
import sys
sys.path.append('../my_modules') # to tell where to find local modules

# 2. Third-party library imports
import pandas as pd
from rdkit import Chem
from rdkit.Chem import (
    Draw,
    PandasTools
)
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
PandasTools.RenderImagesInAllDataFrames(images=True) # to molecules as images in DataFrames
from rdkit.Chem.Draw import IPythonConsole # needed to show molecules
# from rdkit.Chem.Draw.MolDrawing import MolDrawing, DrawingOptions # only needed if modifying defaults
from tqdm.auto import tqdm

# 3. Local application imports
import kernel_infos

In [ ]:
# Information about the kernel
kernel_infos.show_kernel_info()

In [ ]:
# Global variables
HERE = Path().resolve()
print(f'{HERE}')
ROOT = HERE.parent
print(f'{ROOT}')
DATA = ROOT / 'data'
print(f'{DATA}')

## Load previous data

In [ ]:
# File to load
EGFR_compounds_lipinski_csv_path = DATA / "EGFR_compounds_lipinski.csv"

In [ ]:
# Load in a dataframe and display the first 3 rows
egfr_df = pd.XXX(EGFR_compounds_lipinski_csv_path, index_col=0)
print(f"Dataframe shape: {egfr_df.shape}")
egfr_df.XXX

In [ ]:
# Drop descriptor (MW, HBA, HBD and LogP) columns
egfr_df.drop(columns=[XXX], inplace=True)
print(f"Dataframe shape: {egfr_df.shape}")
egfr_df.head(3)

In [ ]:
# Add molecule column: ROMol
PandasTools.XXX(egfr_df, smilesCol='smiles')
egfr_df.head(3)

## Filter for PAINS using RDKit implemented solution

In [ ]:
# Initialize the FilterCatalog class for PAINS 
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.XXX)
catalog = FilterCatalog(params)

In [ ]:
# Search for PAINS
# Initialize 2 lists
pains_l = XXX
no_pains_l = XXX

for index, row in tqdm(egfr_df.iterrows(), total = egfr_df.shape[0]):
    entry = catalog.GetFirstMatch(row['ROMol']) # Get the first matching PAINS
    if entry is not None:
        # store PAINS information as a dictionnary
        pains_l.append(
            {
                "ChEMBL_ID": row[XXX],
                "rdkit_molecule": row[XXX],
                'pains': entry.GetDescription().capitalize()
            }
        )
    else:
        # Collect indices of molecules without PAINS
        no_pains_l.XXX(index)

In [ ]:
# Transform pains_l in a dataframe
pains_df = pd.XXX(pains_l)

# How many compounds ?
print(f"Number of coumpounds with a PAINS substructure: {pains_df.XXX}")

# Display the first 3 lignes of the dataframe
pains_df.XXX

In [ ]:
# Draw an image with a list of the first 3 compounds and add the ChEML_ID and the name of the detected PAINS to the legend
XXX.MolsToGridImage(
    pains_df['rdkit_molecule'].head(3).XXX,
    legends = [f"{id}: {name}" for id, name in XXX(pains_df['ChEMBL_ID'], pains_df['pains'])]
)

In [ ]:
# How many compounds without PAINS substructure ? (use a "fancy indexing": a list indexation)
print(f"Number of compounds without PAINS substructure: {egfr_df.loc[XXX].shape[0]}")

## Filter for BRENK unwanted substructures
BRENK substructures are now implemented in RDKit but use the provided list of the supporting information from Brenk _et al._ (https://doi.org/10.1002/cmdc.200700139
) as an external list to get the substructure matches manually

### Filtering

In [ ]:
# Brenk unwanted substructures file path
brenk_csv_path = DATA / 'unwanted_Brenk_substructures.csv'

In [ ]:
# Load the Brenk unwanted substructures in a dataframe
# Separator is a space character
brenk_df = pd.XXX(brenk_csv_path, XXX=' ')

In [ ]:
# How many structures
print(f'Number of Brenk unwanted substructures: {XXX.shape[0]}')

In [ ]:
# Head the dataframe
brenk_df.head(3)

In [ ]:
# Add a 'rdkit_molecule' column from its 'smarts' code to brenk_df
brenk_df['rdkit_molecule'] = brenk_df[XXX].map(Chem.XXX)

In [ ]:
# Draw an image with the first 3 molecules corresponding to the structure with their #name
n = 3
Draw.XXX(
    molsPerRow = 3,
    XXX = brenk_df.head(n)['rdkit_molecule'].tolist(),
    legends= brenk_df.head(n)['#name'].tolist()
)

In [ ]:
egfr_df.head(1)

In [ ]:
# Search egfr_df for unwanted substructures
brenk_l = list()
no_brenk_l = list()

for index, row in tqdm(egfr_df.iterrows(), total = egfr_df.shape[0]):
    match = False
    for _, substructure in brenk_df.iterrows():
        if row['ROMol'].HasSubstructMatch(substructure['rdkit_molecule']):
            brenk_l.append(
                {
                    'ChEMBL_ID': row['molecule_chembl_id'],
                    'rdkit_molecule': row['ROMol'],
                    'substructure': substructure['rdkit_molecule'],
                    'substructure_name': substructure['#name'],
                }                
            )
            match = True
    if not match:
        # Collect indices of molecules without Brenk substructure
        no_brenk_l.append(index)

In [ ]:
# Transform brenk_l in a dataframe
brenk_df = XXX.DataFrame(brenk_l)

# How many compounds ?
print(f"Number of coumpounds with a Brenk substructure: {XXX}")

# Show the first 3 lignes of the dataframe
brenk_df.XXX

In [ ]:
# How many compounds without Brenk substructure ? (use a "fancy indexing": a list indexation)
print(f"Number of compounds without Brenk substructure: {egfr_df.loc[no_brenk_l].shape[0]}")

### Highlight Brenk substructures in an image of the first 3 compounds containing a Brenk substructures

In [ ]:
# number of molecules to highlight
n=3

# list of molecules
mols = brenk_df.head(XXX)['rdkit_molecule'].XXX

# list of atom lists
to_highlight = [row['rdkit_molecule'].GetSubstructMatch(row['substructure']) for _, row in brenk_df.head(XXX).iterrows()]

# list of legends
legends = brenk_df.head(n)['substructure_name'].XXX

# image
Draw.MolsToGridImage(
    mols,
    molsPerRow=3,
    subImgSize=(300, 300),
    highlightAtomLists=to_highlight,
    legends = XXX
)

### Most frequent Brenk substructures

In [ ]:
# Define groups by "substructure_name"
groups = brenk_df.groupby(XXX)

# Compute group frequencies
group_frequencies = groups.size()

# Sort them
group_frequencies.XXX(ascending=False, inplace=True)

# Print the 10 first frequencies
n = XXX
group_frequencies.head(n)